In [1]:
# Libraries
import pandas as pd
import numpy as np
import clip
import torch
from PIL import Image
import pickle
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
import google.genai as genai
import os
from dotenv import load_dotenv

c:\porsche-appreciation-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set up vector space and AI model
# note: always restart this kernel if you want to re-run this cell !!!

# .env file
load_dotenv()
# for ai
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# CLIP model
clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)

# Text model
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Vector space
client = QdrantClient(path='../data/qdrant_db')
image_collection_name = 'porsche_images'
text_collection_name = 'porsche_text'

# Appreciation prediction model
with open('../data/porsche_model.pkl', 'rb') as f:
    appreciation_model = pickle.load(f)
# OH Encoder
with open('../data/porsche_encoder.pkl', 'rb') as f:
    appreciation_encoder = pickle.load(f)

# Historical matcher - find old value of user's car based on features
import sys
sys.path.insert(0, os.path.abspath('../..'))                                            # project root — historical_matcher.py and its dependencies now live here
from historical_matcher import HistoricalMatcher
historical_matcher = HistoricalMatcher()
historical_matcher.load_historical_listings_from_csv()

# Gemini AI
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables")
GEMINI_MODEL = "gemini-2.0-flash"
gemini_model = genai.Client(api_key=GEMINI_API_KEY)

In [11]:
# New method: Based on user's image, find similar listings in vector space
def find_similar_listings(image_path, top_k=10):
    # Encode user's image
    image = Image.open(image_path).convert('RGB')
    image_tensor = clip_preprocess(image).unsqueeze(0).to(device)                       # call CLIP preprocessing model
    with torch.no_grad():                                                               # no_grad: remove gradient computation, not needed because only doing forward passes (not training)
        image_features = clip_model.encode_image(image_tensor)                          # call CLIP encoding model
        image_features = image_features / image_features.norm(dim=1, keepdim=True)

    # Compare user's image with images in Qdrant (vector space)
    search_results = client.query_points(
        collection_name=image_collection_name,
        query=image_features.cpu().numpy()[0].tolist(),                                 # convert image_features (vector of user's image) into list item
        limit=top_k                                                                     # show only top 10 most similar image matches
    )
    similar_listings = []

    for result in search_results.points:                                                # gather all similar images & their features
        similar_listings.append({
            'listing_id': result.payload['listing_id'],
            'model_year': result.payload['model_year'],
            'model_type': result.payload['model_type'],
            'mileage': result.payload['mileage'],
            'condition': result.payload['condition'],
            'price_now': result.payload['price_now'],
            'similarity_score': result.payload,
            'source': result.payload['source']
        })
    return similar_listings

In [12]:
# Aggregate output from similar_listings

def extract_vehicle_info(similar_listings):
    """
    Get model_year, model_type, condition. Use 'voting' from top matches
    """
    if not similar_listings:
        return None
    from collections import Counter                                                     # to weigh by similarity score

    # Get top 5 most similar
    top_5 = similar_listings[:5]

    # model_type: most common
    model_types = [l['model_type'] for l in top_5]
    model_type = Counter(model_types).most_common(1)[0][0]

    # model_year: median of top 5
    years = [float(l['model_year']) for l in top_5 if pd.notna(l['model_year'])]
    model_year = int(np.median(years)) if years else None                               # only if there is data for model_year

    # condition: most common
    conditions = [l['condition'] for l in top_5]
    condition = Counter(conditions).most_common(1)[0][0]

    # mileage: median (future modification: better if user-provided)
    mileages = [l['mileage'] for l in top_5 if pd.notna(l['mileage'])]
    mileage = int(np.median(mileages)) if mileages else None                            # only if there is data for mileage

    return {
        'model_type': model_type,
        'model_year': model_year,
        'condition': condition,
        'mileage': mileage,
        'similar_listings': similar_listings,
    }
    

In [5]:
# Based on predicted car data, get price from 3 years ago

def get_historical_price(vehicle_info):
    target_listing = {
        'model_year': str(vehicle_info['model_year']),
        'model_type': vehicle_info['model_type'],
        'mileage': vehicle_info.get('mileage',''),
        'condition': vehicle_info['condition']
    }

    price_3_years_ago = historical_matcher.calculate_price_3_years_ago(target_listing)
    return price_3_years_ago

In [19]:
# LLM valuation reasoning with Gemini

def generate_valuation_with_llm(image_path, vehicle_info, similar_listings, price_3_years_ago, current_price=None):
    # Prepare summary on similar listings
    similar_summary = '\n'.join([
        f"- {l['model_type']} {l['model_year']} ({l['condition']}, {l['mileage']}mi, USD{l['price_now']:,})"
        for l in similar_listings[:5]
    ])

    prompt = f"""
    You are an expert Porsche appraiser with 20 years of experience. Analyse this Porsche listing and provide a market valuation.

    Vehicle Information (from image matching):
    - Model Type: {vehicle_info['model_type']}
    - Model Year: {vehicle_info['model_year']}
    - Condition: {vehicle_info['condition']}
    - Mileage: {vehicle_info.get('mileage', 'Unknown')}

    Similar Historical Listings (from vector search):
    {similar_summary}

    Historical Price (3 years ago): {"${:,}".format(price_3_years_ago) if price_3_years_ago else "Unknown"} (if available)

    Current Asking Price: {"${:,}".format(current_price) if current_price else "Not provided"}

    Analyse the uploaded image, compare with similar listings, and provide:
    1. Estimated Current Market Value (USD)
    2. Brief reasoning (2-3 sentences)
    3. Key factors affecting valuation

    Format your response as:
    VALUATION: $X,XXX,XXX
    REASONING: [your analysis]
    FACTORS: [key factors]
    """

    # Load and prep image
    image = Image.open(image_path)
    response = gemini_model.models.generate_content(model=GEMINI_MODEL, contents=[prompt, image])

    return response.text

In [25]:
# Predict future appreciation

def predict_appreciation(vehicle_info, current_valuation, price_3_years_ago):
    # Prep features
    from sklearn.preprocessing import OneHotEncoder, LabelEncoder

    input_df = pd.DataFrame([{
        'model_year': vehicle_info['model_year'],
        'model_type': vehicle_info['model_type'],
        'mileage': vehicle_info['mileage'],
        'condition': vehicle_info['condition'],
        'price_now': current_valuation,
        'price_3_years_ago': price_3_years_ago if price_3_years_ago is not None else 0,
    }])

    # Load encoder used during training
    # Use feature_names_in_ to match the exact columns the encoder was fit on
    # (price_3_years_ago was dtype object in the training CSV, so it was OHE-encoded too)
    object_cols = appreciation_encoder.feature_names_in_.tolist()
    for col in object_cols:
        input_df[col] = input_df[col].astype(str)

    # Apply same pre-processing as training
    input_oh = pd.DataFrame(appreciation_encoder.transform(input_df[object_cols]))
    input_oh.index = input_df.index
    num_input = input_df.drop(object_cols, axis=1)
    X_input = pd.concat([num_input, input_oh], axis=1)
    X_input.columns = X_input.columns.astype(str)

    # Return prediction
    raw_pred = appreciation_model.predict(X_input)
    label = int(np.round(raw_pred[0]))

    return {
        'will_appreciate': bool(label == 1),
        'confidence': round(float(raw_pred[0]), 2)
    }

    # WITH VALUE
    # appreciation_pct = round(
    #     (current_valuation - price_3_years_ago) / price_3_years_ago * 100, 1
    # )
    # return {
    #     'will_appreciate': 'yes' if label == 1 else 'no',
    #     'appreciation_pct_3yr': appreciation_pct,
    #     'price_3_years_ago': price_3_years_ago,
    #     'current_price': current_valuation,
    # }

In [8]:
# Extract and parse LLM valuation
import re

def parse_llm_valuation(llm_response):
    match = re.search(r'VALUATION:\s*\$?([\d,]+)', llm_response)
    if match:
        return float(match.group(1).replace(',', ''))
    return None

In [28]:
# Complete valuation pipeline
def full_valuation_pipeline(image_path, current_price=None):
    # get user image -> match with images in vector space -> get info of top results -> predict valuation in 3 yrs -> output whether appreciated or not

    print('Step 1: Finding similar listings ...')
    similar_listings = find_similar_listings(image_path, top_k=10)
    
    print('Step 2: Extracting car information ...')
    vehicle_info = extract_vehicle_info(similar_listings)
    print(f'Detected: {vehicle_info["model_type"]} {vehicle_info["model_year"]} {vehicle_info["condition"]}')

    print('Step 3: Getting historical price ...')
    price_3_years_ago = get_historical_price(vehicle_info)
    price_3_years_ago = None if price_3_years_ago == 'insufficient_data' else price_3_years_ago
    print(f'Price 3 years ago: {"USD{:,}".format(price_3_years_ago) if price_3_years_ago else "insufficient data"}')

    print('Step 4: Generating LLM valuation ...')
    llm_valuation = generate_valuation_with_llm(image_path, vehicle_info, similar_listings, price_3_years_ago, current_price)

    print('Step 5: Predicting 3-year appreciation ...')
    current_valuation = parse_llm_valuation(llm_valuation)
    if current_valuation is None:
        # use current_price or median of simiilar listings
        current_valuation = current_price or np.median([l['price_now'] for l in similar_listings[:5]])
    appreciation_pred = predict_appreciation(vehicle_info, current_valuation, price_3_years_ago)

    return {
        'vehicle_info': vehicle_info,
        'similar_listings': similar_listings,
        'price_3_years_ago': price_3_years_ago,
        'llm_valuation': llm_valuation,
        'current_valuation': current_valuation,
        'appreciation_prediction': appreciation_pred,
    }

In [27]:
# test
result = full_valuation_pipeline('C:\porsche-appreciation-predictor\project\data\images\p00001_image_1.jpg')
print(result)

Step 1: Finding similar listings ...
Step 2: Extracting car information ...
Detected: 944 1988 Fair
Step 3: Getting historical price ...
Price 3 years ago: insufficient data
Step 4: Generating LLM valuation ...
Step 5: Predicting 3-year appreciation ...
{'vehicle_info': {'model_type': '944', 'model_year': 1988, 'condition': 'Fair', 'mileage': 40000, 'similar_listings': [{'listing_id': 'p00001', 'model_year': 1987.0, 'model_type': '944', 'mileage': 132000.0, 'condition': 'Fair', 'price_now': 16250.0, 'similarity_score': {'listing_id': 'p00001', 'model_year': 1987.0, 'model_type': '944', 'mileage': 132000.0, 'condition': 'Fair', 'price_now': 16250.0, 'source': 'https://bringatrailer.com/listing/1987-porsche-944-turbo-167/'}, 'source': 'https://bringatrailer.com/listing/1987-porsche-944-turbo-167/'}, {'listing_id': 'p00998', 'model_year': 1988.0, 'model_type': '928', 'mileage': 64000.0, 'condition': 'Fair', 'price_now': 16250.0, 'similarity_score': {'listing_id': 'p00998', 'model_year': 1